Here is the code where you put in H, M, G, model and measurement

In [49]:
#| default_exp prediction_module

In [37]:
#| export
import pickle
import pandas as pd 
from anthropmass.data_module import *
from anthropmass.anthro_module import *
from anthropmass.bambi_module import *

Normalize is in data module but i dont know how to import

This function is normalizing the persons weight and height

In [38]:
#| export
def normalize_person(weight, height, gender):
    person=pd.DataFrame({'weightkg': [weight], 'stature': [height], 'Gender': [gender]})
    normalize(person, 'weightkg')
    normalize(person, 'stature')
    return person

This functions gets the pickled model

In [39]:
#| export
def get_pickled_model(kindofmodel:str, measurement:str):
    filepath = f'../output/anthro_models/{kindofmodel}/pickle_{measurement}_{kindofmodel}'
    with open(filepath,'rb') as file:
        model=pickle.load(file)
    return model

Predict_from_model uses the pickled model and the normalized person to predict the measurement

In [ ]:
#| export
def predict_from_model(kindofmodel:str, measurement:str, w, h, g):
    pickledmodel = get_pickled_model(kindofmodel, measurement)
    person = normalize_person(w, h, g)
    if kindofmodel=='xgboost':
        return pickledmodel.predict(person)
    elif kindofmodel=='bambi':
        model = model_bmb(measurement)
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    elif kindofmodel=='bambi_new':
        #train=pd.read_csv('../data/processed/ANSURIInormalizedtrain.csv')
        model = component_model(measurement,train)
        person['Component']='Army Reserve' #'Regular Army'#'Army National Guard'#'Army Reserve'
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    elif kindofmodel=='bambi_ml_gc':
        train=pd.read_csv('../data/processed/ANSURIInormalizedtrain.csv')
        model = component_model(measurement,train)
        person['Component']='Army Reserve' #'Regular Army'#'Army National Guard'#'Army Reserve'
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    else:
        return 'wrong model name'

In [41]:
#| export
def add_to_df(df, measurement, pred):
    df[measurement]=pred
    return df

I dont know if we need to make it to csv?

In [100]:
#| export
def make_csv(data, name):
    data.to_csv(f'{name}.csv', index=False)

Here are all predictions made, the measurements are in a list

In [43]:
#| export
def make_predictions(kindofmodel:str, measurements:list, w, h, g):
    output=pd.DataFrame()
    for m in measurements:
        pred = predict_from_model(kindofmodel, m, w, h, g)
        add_to_df(output, m, pred)
    return output

In [56]:
make_predictions('bambi',['abdominalextensiondepthsitting','acromialheight'], 80, 1700, 1)

,abdominalextensiondepthsitting,acromialheight
0,250.066345,1390.5659


In [81]:
make_predictions('xgboost',['abdominalextensiondepthsitting','acromialheight','neckcircumference'], 81.5, 1776, 1)

,abdominalextensiondepthsitting,acromialheight,neckcircumference
0,252.47998,1457.88855,384.442688


In [119]:
test=pd.read_csv('../data/processed/ANSURIItest.csv')
test_preds_all=pd.DataFrame()
variables = test.iloc[:,1:94].drop('weightkg',axis=1).drop('stature',axis=1)
variables=variables[:2]

for index, row in test.iterrows():
    test_pred_xgb = make_predictions('xgboost',variables, row['weightkg'], row['stature'], row['Gender'])
    test_preds_all = pd.concat([test_preds_all, test_pred_xgb], ignore_index=True)
    if index == 2:
        break
test_preds_all

KeyboardInterrupt: 

In [ ]:
make_csv(test_preds_all,'../output/anthro_results/predict_test_xgboost')

In [50]:
import nbdev; nbdev.nbdev_export()